In [2]:
import pandas as pd
import numpy as np
import re
import ast

In [6]:
df = pd.read_csv("movies_dataset.csv")
keep_cols = ["id", "title", "release_date", "budget", "revenue", "genres"]
df = df[keep_cols].copy()
df.head()


,id,title,release_date,budget,revenue,genres
0,475762,Return to Justice,1990-01-01,0.0,0.0,"['Drama', 'Action']"
1,42447,Cyber-C.H.I.C.,1990-08-09,0.0,0.0,"['Action', 'Comedy']"
2,769,GoodFellas,1990-09-12,25000000.0,47072327.0,"['Drama', 'Crime']"
3,51329,There Were Days... and Moons,1990-04-11,0.0,0.0,"['Drama', 'Comedy']"
4,258381,Working Tra$h,1990-11-26,0.0,0.0,"['Comedy', 'TV Movie']"


In [7]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

In [8]:
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")

In [9]:
def extract_genre_names(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [g.get("name") for g in x if isinstance(g, dict) and "name" in g]
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return [g.get("name") for g in parsed if isinstance(g, dict) and "name" in g]
        except Exception:
            return []
    return []

df["genre_list"] = df["genres"].apply(extract_genre_names)
df["main_genre"] = df["genre_list"].apply(lambda x: x[0] if len(x) > 0 else np.nan)

In [10]:
df["profit"] = df["revenue"] - df["budget"]
df["roi"] = np.where(df["budget"] > 0, df["revenue"] / df["budget"], np.nan)

In [11]:
bins = [1989, 1999, 2009, 2019, 2025]
labels = ["1990s", "2000s", "2010s", "2020-2025"]
df["release_period"] = pd.cut(df["release_year"], bins=bins, labels=labels)

In [12]:
df_rq1 = df[
    (df["release_year"].between(1990, 2025, inclusive="both")) &
    (df["budget"] > 0) &
    (df["revenue"] > 0)
].copy()

In [13]:
df_rq1.shape

(2373, 12)

In [14]:
movies_per_year = df_rq1.groupby("release_year").size()
movies_per_year.describe()

count    35.000000
mean     67.800000
std      14.748479
min      34.000000
25%      58.000000
50%      69.000000
75%      80.500000
max      93.000000
dtype: float64

In [15]:
df_rq1[["budget", "revenue", "profit", "roi"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,budget,revenue,profit,roi
count,2.373000e+03,2.373000e+03,2.373000e+03,2373.000000
mean,5.247649e+07,1.784184e+08,1.259419e+08,11.658646
std,5.863897e+07,2.697628e+08,2.304580e+08,278.102133
min,1.000000e+00,1.400000e+01,-2.190078e+08,0.000021
1%,1.000000e+05,2.459968e+04,-5.126877e+07,0.010169
5%,1.800000e+06,6.342608e+05,-2.137153e+07,0.103005
50%,3.000000e+07,7.805314e+07,4.150000e+07,2.506418
95%,1.800000e+08,7.402472e+08,5.804376e+08,13.026489
99%,2.500000e+08,1.249635e+09,1.035646e+09,34.399815
max,4.899000e+08,2.923706e+09,2.686706e+09,12890.386667


In [16]:
df_rq1.sort_values("roi", ascending=False)[["title", "release_year", "budget", "revenue", "roi"]].head(10)
df_rq1.sort_values("profit", ascending=False)[["title", "release_year", "budget", "revenue", "profit"]].head(10)

,title,release_year,budget,revenue,profit
7593,Avatar,2009,237000000.0,2.923706e+09,2.686706e+09
11600,Avengers: Endgame,2019,356000000.0,2.799439e+09,2.443439e+09
2797,Titanic,1997,200000000.0,2.264162e+09,2.064162e+09
12784,Avatar: The Way of Water,2022,350000000.0,2.353096e+09,2.003096e+09
10037,Star Wars: The Force Awakens,2015,245000000.0,2.068224e+09,1.823224e+09
11188,Avengers: Infinity War,2018,300000000.0,2.052415e+09,1.752415e+09
12387,Spider-Man: No Way Home,2021,200000000.0,1.921847e+09,1.721847e+09
9996,Jurassic World,2015,150000000.0,1.671537e+09,1.521537e+09
13591,Inside Out 2,2024,200000000.0,1.698864e+09,1.498864e+09
11905,The Lion King,2019,260000000.0,1.662021e+09,1.402021e+09
